# 04. Algorithmic Logic: Backtracking Sudoku Solver with MCV Heuristics

### Overview
Sudoku is an **NP-complete Constraint Satisfaction Problem (CSP)**. 
A naive backtracking search tests cells from top-left to bottom-right, taking thousands or millions of iterations on complex puzzles.

### Most Constrained Variable (MCV) Heuristic
Instead of scanning in fixed order, we select the cell with the **minimum remaining legal options (fewest candidates)**:
- If an empty cell has only 1 legal candidate, we fill it immediately.
- If an empty cell has 0 legal candidates, we prune the search branch instantly.
This heuristic reduces the search tree depth exponentially, typically solving 9×9 puzzles in $< 10$ milliseconds.

In [ ]:
import os
import sys
import time
import copy
import numpy as np
from typing import List, Tuple, Optional
from pathlib import Path

def find_root() -> Path:
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / "models" / "digit_classifier.keras").exists():
            return parent
    return Path(os.path.abspath("..")).resolve()

PROJECT_ROOT = find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.solver_service import solve_sudoku, find_conflicts, validate_grid

print("Imported solver service successfully.")

## 1. Benchmark Puzzles
We define representative puzzles of varying difficulty, including an intentionally invalid/contradictory puzzle to test rule verification.

In [ ]:
SAMPLE_PUZZLE = [
    [5, 3, 0,  0, 7, 0,  0, 0, 0],
    [6, 0, 0,  1, 9, 5,  0, 0, 0],
    [0, 9, 8,  0, 0, 0,  0, 6, 0],

    [8, 0, 0,  0, 6, 0,  0, 0, 3],
    [4, 0, 0,  8, 0, 3,  0, 0, 1],
    [7, 0, 0,  0, 2, 0,  0, 0, 6],

    [0, 6, 0,  0, 0, 0,  2, 8, 0],
    [0, 0, 0,  4, 1, 9,  0, 0, 5],
    [0, 0, 0,  0, 8, 0,  0, 7, 9]
]

def format_grid(grid: List[List[int]]) -> str:
    """Pretty print a 9x9 Sudoku grid with 3x3 block dividers."""
    lines = []
    for r in range(9):
        if r > 0 and r % 3 == 0:
            lines.append("------+-------+------")
        row_str = []
        for c in range(9):
            if c > 0 and c % 3 == 0:
                row_str.append("|")
            val = grid[r][c]
            row_str.append(str(val) if val != 0 else ".")
        lines.append(" ".join(row_str))
    return "\n".join(lines)

print("Original Puzzle:\n")
print(format_grid(SAMPLE_PUZZLE))

## 2. Solving with Search Node Tracking
Run the solver on the sample puzzle and inspect execution time and total search tree nodes explored.

In [ ]:
success, solved, nodes, time_ms, _ = solve_sudoku(SAMPLE_PUZZLE)

print(f"Solved: {success} | Nodes Explored: {nodes} | Duration: {time_ms:.2f} ms\n")
print("Solution Grid:\n")
print(format_grid(solved))

## 3. Contradiction & Conflict Detection
A robust solver must detect contradictory grids (e.g. duplicate numbers in a row, col, or 3×3 block) before searching.

In [ ]:
invalid_puzzle = copy.deepcopy(SAMPLE_PUZZLE)
# Create conflict: duplicate 5 in first row
invalid_puzzle[0][4] = 5

conflicts = find_conflicts(invalid_puzzle)
print(f"Detected {len(conflicts)} rule violations:")
for c in conflicts:
    print(f"  Conflict at row {c['row']}, col {c['col']}: Value {c['val']} ({c['reason']})")

success, _, nodes, _, _ = solve_sudoku(invalid_puzzle)
print(f"\nSolver correctly rejected invalid puzzle: {not success}")

## 4. Benchmark Across 100 Runs
Evaluating solver consistency and throughput.

In [ ]:
times = []
for _ in range(100):
    start = time.perf_counter()
    solve_sudoku(SAMPLE_PUZZLE)
    times.append((time.perf_counter() - start) * 1000)

print(f"Benchmark (100 runs):")
print(f"  Mean Duration:   {np.mean(times):.2f} ms")
print(f"  Median Duration: {np.median(times):.2f} ms")
print(f"  Min Duration:    {np.min(times):.2f} ms")
print(f"  Max Duration:    {np.max(times):.2f} ms")